# MTG Image Recognition

Projekat iz **Mašinskog učenja**.
Cilj: naučiti model da prepozna kartu slikanu kamerom, na osnovu treninga nad slikama velike rezolucije koje su prethodno pripremljene augmentacijom podataka.

## Podešavanje okruženja

Pokrećemo iz root-a repozitorijuma (`MTG_image_recognition`) i učitavamo module iz `src/`.

In [ ]:
import os
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from src.augment import augment_image, augment_image_x_times
from src.helper import (
    encode_images,
    load_images_from_folder,
    save_encoded_images,
    save_pil_images,
)
from src.build_dataset import CARD_LABELS, build_dataset

random.seed(42)

##  Cilj projekta

> Naučiti model da prepoznaje kartu slikanu kamerom na osnovu treninga nad slikama velike rezolucije (prethodno pripremljenim za treniranje raznim metodama augmentacije podataka)

Ukratko: prepoznavanje jedne od **5 kartica** (5 klasa) sa **slike napravljene telefonom**, pri čemu se model trenira na **augmentovanim verzijama** visoko-rezolutivnih skenova.

## Podaci

Bazne slike („raw", originalni skenovi) su u `data/raw/<naziv karte>/`.
Svaka klasa ima jednu visoko-rezolutivnu sliku karte.

In [ ]:
fig, axes = plt.subplots(1, len(CARD_LABELS), figsize=(3 * len(CARD_LABELS), 4))

for ax, (card, label) in zip(axes.flat, CARD_LABELS.items()):
    raw_images = load_images_from_folder(f"data/raw/{card}")
    ax.imshow(raw_images[0])
    ax.set_title(f"{card}\nlabela: {label}")
    ax.axis("off")

plt.tight_layout()
plt.show()

## Augmentacija

Augmentacija simulira razlike u kadriranju, položaju i osvetljenju koje nastanu kada se karta slika telefonom.
`src/augment.py` primenjuje nasumično:

- horizontalni flip (50% šanse)
- rotaciju +- 15 stepeni
- promenu osvetljenja (0.8–1.2×)
- promenu kontrasta (0.8–1.2×)

In [ ]:
img = load_images_from_folder("data/raw/angelic_renewal")[0]
augmented = augment_image_x_times(img, 5)

fig, axes = plt.subplots(1, 6, figsize=(3 * 6, 4))
axes[0].imshow(img)
axes[0].set_title("Original")
axes[0].axis("off")

for ax, aug in zip(axes[1:], augmented):
    ax.imshow(aug)
    ax.axis("off")

plt.tight_layout()
plt.show()

## Pipeline podataka

`src/build_dataset.py`

Za svaku kartu:
1. učitava raw slike (`load_images_from_folder`)
2. pravi **10 augmentovanih kopija** po slici (`augment_image_x_times`)
3. čuva kopije u punoj rezoluciji u `data/augmented/<karta>/`
4. enkoduje (resize na 32×32, normalizacija, spljošteno u 1D niz) i čuva u `data/encoded/<karta>/`
5. na kraju čuva `data/encoded/X.npy` i `data/encoded/y.npy`

In [ ]:
X, y = build_dataset()

print(f"X shape: {X.shape}   (broj primera, 32*32*3)")  # 50 = 5 klasa x 10 augmentacija
print(f"y shape: {y.shape}   (labela 0-4 za svaku sliku)")
print(f"Raspodela po klasama:")
for card, label in CARD_LABELS.items():
    print(f"  {label}: {card}, {np.sum(y == label)} primera")

In [ ]:
# Pregled enkodovanih slika (32x32) iz foldera raw
fig, axes = plt.subplots(1, len(CARD_LABELS), figsize=(3 * len(CARD_LABELS), 3))

for ax, (card, label) in zip(axes.flat, CARD_LABELS.items()):
    raw = load_images_from_folder(f"data/raw/{card}")[0]
    encoded = encode_images([raw])[0].reshape(32, 32, 3)
    ax.imshow(encoded)
    ax.set_title(f"{card} (labela {label})")
    ax.axis("off")

plt.tight_layout()
plt.show()

## 5. Naredni koraci (TODO)

Deo pipelajna koji sledi još **nije implementiran** — ovo su predloženi koraci za finalnu verziju:

1. **Model** — definisati arhitekturu (npr. malu CNN mrežu ili transfer learning sa pretreniranim modelom, npr. ResNet18). Ulaz su enkodovane slike iz `X.npy`.
2. **Treniranje** — podeliti podatke na train/validation test, definisati loss i optimizer, trenirati i čuvati težine u `models/`.
3. **Evaluacija** — metrike (accuracy, confusion matrix) nad test skupom.
4. **Predikcija** — funkcija koja prima sliku napravljenu telefonom i vraća naziv/klasu karte.
5. **Demonstracija** — nekoliko pravih fotografija karata iz telefona na kojima model daje rezultat.

In [ ]:
# TODO primer: priprema podataka za treniranje jednim pozivom
#
# x_train, x_val, y_train, y_val = train_test_split(
#     X.reshape(-1, 32, 32, 3), y, test_size=0.2, random_state=42, stratify=y
# )
#
# Ostatak (model, trening, evaluacija, predikcija) se implementira u narednim koracima.
print("Podaci su spremni:", X.shape, "— sledeći korak: model.")